# Clinical Decision Support System for Heart Failure Mortality Risk Prediction

**Project goal:** Predict heart failure `DEATH_EVENT` using a SMOTE-enhanced tuned Stacking Ensemble model.

**Dataset:** [Kaggle Heart Failure Clinical Data](https://www.kaggle.com/datasets/andrewmvd/heart-failure-clinical-data/data)

**Final method used:** Tuned Stacking Ensemble with SMOTE, cross-validation, and a Random Forest meta-model.

The stacking model combines Logistic Regression, Random Forest, XGBoost, and SVM as base learners. A Random Forest meta-model then learns how to combine their probability predictions.


## 1. Import Libraries

If any package is missing, run this once:

```python
# !pip install -r requirements.txt
```


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
TARGET_COLUMN = "DEATH_EVENT"
TIME_COLUMN = "time"


## 2. Load Dataset

The notebook looks for `heart_failure_prediction.csv` first, then the original Kaggle filename `heart_failure_clinical_records_dataset.csv`.


In [ ]:
candidate_paths = [
    Path("heart_failure_prediction.csv"),
    Path("heart_failure_clinical_records_dataset.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Dataset CSV not found in the current folder.")

df = pd.read_csv(data_path)
print(f"Loaded dataset: {data_path}")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


In [ ]:
df.info()


In [ ]:
df.isna().sum().sort_values(ascending=False)


## 3. Exploratory Data Analysis


In [ ]:
target_counts = df[TARGET_COLUMN].value_counts().sort_index()
target_summary = pd.DataFrame({
    "class": ["Survived", "Death Event"],
    "count": [target_counts.get(0, 0), target_counts.get(1, 0)],
    "percentage": [
        target_counts.get(0, 0) / len(df) * 100,
        target_counts.get(1, 0) / len(df) * 100,
    ],
})
target_summary


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET_COLUMN, hue=TARGET_COLUMN, palette="Set2", legend=False)
plt.xticks([0, 1], ["Survived", "Death Event"])
plt.title("Target Distribution")
plt.xlabel("Outcome")
plt.ylabel("Patient Count")
plt.show()


In [ ]:
df.describe().T


In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Feature Correlation Heatmap")
plt.show()


## 4. Feature and Target Preparation

The `time` column is excluded by default because it represents follow-up duration. For a baseline clinical decision support system, using it can cause target leakage.


In [ ]:
INCLUDE_TIME = False

drop_columns = [TARGET_COLUMN]
if not INCLUDE_TIME:
    drop_columns.append(TIME_COLUMN)

X = df.drop(columns=drop_columns)
y = df[TARGET_COLUMN].astype(int)

print("Features used:")
print(list(X.columns))
print(f"\nTarget: {TARGET_COLUMN}")
print(f"Include time feature: {INCLUDE_TIME}")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows: {X_test.shape[0]}")


## 5. Build Base Learners for Stacking

These base models are not used as separate final models. They are components of the Stacking Ensemble. SMOTE is included inside each pipeline so oversampling happens only during training folds.


In [ ]:
def make_scaled_classifier(classifier):
    return ImbPipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("smote", SMOTE(random_state=RANDOM_STATE)),
            ("classifier", classifier),
        ]
    )


def make_tree_classifier(classifier):
    return ImbPipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("smote", SMOTE(random_state=RANDOM_STATE)),
            ("classifier", classifier),
        ]
    )


negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / max(positive_count, 1)

base_models = {
    "Logistic Regression": make_scaled_classifier(
        LogisticRegression(
            max_iter=2000,
            solver="liblinear",
            random_state=RANDOM_STATE,
        )
    ),
    "Random Forest": make_tree_classifier(
        RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
    "XGBoost": make_tree_classifier(
        XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
    "SVM": make_scaled_classifier(
        SVC(
            kernel="rbf",
            probability=True,
            random_state=RANDOM_STATE,
        )
    ),
}

list(base_models.keys())


## 6. Tune Base Learners

The base learners are tuned on the training data only. The main tuning target is F1 score, while recall and precision are tracked to keep the model balanced.


In [ ]:
TUNING_ITERATIONS = 8
TUNING_CV_FOLDS = 3
TUNING_REFIT_METRIC = "f1"
TUNING_SCORING = {
    "f1": make_scorer(f1_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
    "roc_auc": "roc_auc",
}

tuning_cv = StratifiedKFold(
    n_splits=TUNING_CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

search_spaces = {
    "Logistic Regression": {
        "classifier__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10],
        "classifier__penalty": ["l1", "l2"],
    },
    "Random Forest": {
        "classifier__n_estimators": [300, 500, 800],
        "classifier__max_depth": [None, 3, 5, 8],
        "classifier__min_samples_leaf": [1, 2, 4, 6],
        "classifier__max_features": ["sqrt", "log2", None],
    },
    "XGBoost": {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__max_depth": [2, 3, 4],
        "classifier__learning_rate": [0.03, 0.05, 0.1],
        "classifier__subsample": [0.8, 0.9, 1.0],
        "classifier__colsample_bytree": [0.8, 0.9, 1.0],
        "classifier__reg_lambda": [1, 2, 5],
    },
    "SVM": {
        "classifier__C": [0.1, 0.3, 1, 3, 10],
        "classifier__gamma": ["scale", 0.01, 0.03, 0.1, 0.3],
    },
}

tuned_base_models = {}
tuning_rows = []

for model_name, model in base_models.items():
    print(f"Tuning {model_name}...")
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=search_spaces[model_name],
        n_iter=TUNING_ITERATIONS,
        scoring=TUNING_SCORING,
        cv=tuning_cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=TUNING_REFIT_METRIC,
        error_score="raise",
    )
    search.fit(X_train, y_train)
    tuned_base_models[model_name] = search.best_estimator_
    best_index = search.best_index_
    tuning_rows.append({
        "base_model": model_name,
        "best_cv_f1": search.best_score_,
        "best_cv_recall": search.cv_results_["mean_test_recall"][best_index],
        "best_cv_precision": search.cv_results_["mean_test_precision"][best_index],
        "best_cv_roc_auc": search.cv_results_["mean_test_roc_auc"][best_index],
        "best_parameters": search.best_params_,
    })

tuning_results = pd.DataFrame(tuning_rows).sort_values(
    by=["best_cv_f1", "best_cv_recall", "best_cv_precision"], ascending=False
)
tuning_results


## 7. Cross-Validate and Train Final Stacking Ensemble

The tuned base learners are combined using a Random Forest meta-model. The final stack is evaluated with stratified cross-validation on the training data before the untouched test set is used. `passthrough=True` lets the meta-model use both base-model probabilities and original patient features. The decision threshold is selected from out-of-fold training predictions to optimize F1, with recall and precision used as tie-breakers.


In [ ]:
stacking_estimators = [
    ("lr", tuned_base_models["Logistic Regression"]),
    ("rf", tuned_base_models["Random Forest"]),
    ("xgb", tuned_base_models["XGBoost"]),
    ("svm", tuned_base_models["SVM"]),
]

stacking_model = StackingClassifier(
    estimators=stacking_estimators,
    final_estimator=RandomForestClassifier(
        n_estimators=300,
        max_depth=3,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    stack_method="predict_proba",
    cv=5,
    n_jobs=-1,
    passthrough=True,
)

stack_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
stack_scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

stack_cv_scores = cross_validate(
    stacking_model,
    X_train,
    y_train,
    scoring=stack_scoring,
    cv=stack_cv,
    n_jobs=-1,
    error_score="raise",
)

cv_results = pd.DataFrame([
    {
        "metric": metric_name,
        "mean": np.mean(stack_cv_scores[f"test_{metric_name}"]),
        "std": np.std(stack_cv_scores[f"test_{metric_name}"]),
    }
    for metric_name in stack_scoring
])

display(cv_results.round(4))

out_of_fold_probability = cross_val_predict(
    stacking_model,
    X_train,
    y_train,
    cv=stack_cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

threshold_rows = []
for threshold in np.arange(0.10, 0.91, 0.01):
    threshold_prediction = (out_of_fold_probability >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_train, threshold_prediction, zero_division=0),
        "recall": recall_score(y_train, threshold_prediction, zero_division=0),
        "f1": f1_score(y_train, threshold_prediction, zero_division=0),
    })

threshold_results = pd.DataFrame(threshold_rows)
threshold_results = threshold_results.sort_values(
    by=["f1", "recall", "precision"], ascending=False
).reset_index(drop=True)
OPTIMIZED_THRESHOLD = float(threshold_results.loc[0, "threshold"])

print(f"Optimized decision threshold: {OPTIMIZED_THRESHOLD:.2f}")
display(threshold_results.head(10).round(4))

stacking_model.fit(X_train, y_train)
print("Final model trained: SMOTE Tuned Stacking Ensemble with Random Forest meta-model")


## 8. Evaluate Final Model


In [ ]:
THRESHOLD = OPTIMIZED_THRESHOLD

y_probability = stacking_model.predict_proba(X_test)[:, 1]
y_predicted = (y_probability >= THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_predicted).ravel()

metrics_df = pd.DataFrame([
    {
        "model": "SMOTE Tuned Stacking Ensemble",
        "accuracy": accuracy_score(y_test, y_predicted),
        "precision": precision_score(y_test, y_predicted, zero_division=0),
        "recall": recall_score(y_test, y_predicted, zero_division=0),
        "f1": f1_score(y_test, y_predicted, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_probability),
        "average_precision": average_precision_score(y_test, y_probability),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }
])

metrics_df.round(4)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(
    y_test,
    y_probability,
    name="SMOTE Tuned Stacking Ensemble",
    ax=ax,
)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
ax.set_title("ROC Curve: SMOTE Tuned Stacking Ensemble")
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
plt.show()


In [ ]:
matrix = confusion_matrix(y_test, y_predicted)
fig, ax = plt.subplots(figsize=(6, 5))
cm_display = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=["Survived", "Death Event"],
)
cm_display.plot(ax=ax, values_format="d", colorbar=False)
ax.set_title("Confusion Matrix: SMOTE Tuned Stacking Ensemble")
plt.show()


## 9. Final Notebook Report


In [ ]:
project_summary = pd.DataFrame([
    ["Project", "Clinical Decision Support System for Heart Failure Mortality Risk Prediction"],
    ["Dataset", str(data_path)],
    ["Rows", df.shape[0]],
    ["Columns", df.shape[1]],
    ["Target", TARGET_COLUMN],
    ["Final Model", "SMOTE Tuned Stacking Ensemble"],
    ["Base Learners", "Logistic Regression, Random Forest, XGBoost, SVM"],
    ["Meta-Model", "Random Forest"],
    ["Features Used", ", ".join(X.columns)],
    ["Excluded Feature", TIME_COLUMN if not INCLUDE_TIME else "None"],
    ["Survived Cases", int((y == 0).sum())],
    ["Death Event Cases", int((y == 1).sum())],
    ["Decision Threshold", THRESHOLD],
], columns=["Item", "Value"])

display(project_summary)
display(tuning_results.round(4))
display(cv_results.round(4))
display(threshold_results.head(10).round(4))
display(metrics_df.round(4))


## 10. Predict New Patient Risk

Edit the values in `new_patient` to predict a different patient. `mortality_risk_probability` is the estimated probability of `DEATH_EVENT = 1`.


In [ ]:
new_patient = {
    "age": 65,
    "anaemia": 0,
    "creatinine_phosphokinase": 250,
    "diabetes": 1,
    "ejection_fraction": 35,
    "high_blood_pressure": 1,
    "platelets": 263000,
    "serum_creatinine": 1.3,
    "serum_sodium": 136,
    "sex": 1,
    "smoking": 0,
}

new_patient_df = pd.DataFrame([new_patient])
risk_probability = stacking_model.predict_proba(new_patient_df[X.columns])[:, 1]
prediction = (risk_probability >= THRESHOLD).astype(int)

prediction_result = new_patient_df.copy()
prediction_result["mortality_risk_probability"] = risk_probability.round(4)
prediction_result["predicted_death_event"] = prediction
prediction_result


## 11. Conclusion

This project uses a SMOTE-enhanced tuned Stacking Ensemble to predict heart failure mortality risk. The ensemble combines multiple model perspectives and uses a Random Forest meta-model to learn how to combine their predicted probabilities.

For real clinical use, this model would still need external validation, calibration, clinical review, and ethical approval before being used in patient care.
